In [1]:
pip install torch torchvision

In [10]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity
import torch_geometric
from torch_geometric.data import HeteroData
import networkx as nx
import json
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/graphs", exist_ok=True)

In [11]:
configs = [
    {"name": "tiny",   "d_model": 64,  "nhead": 2, "num_layers": 1},
    {"name": "small",  "d_model": 128, "nhead": 4, "num_layers": 2},
    {"name": "medium", "d_model": 256, "nhead": 8, "num_layers": 4},
]

In [12]:
def generate_trace(config, trace_id):
    model = nn.Transformer(
        d_model=config["d_model"],
        nhead=config["nhead"],
        num_encoder_layers=config["num_layers"],
        num_decoder_layers=config["num_layers"]
    )

    src = torch.rand(10, 32, config["d_model"])
    tgt = torch.rand(10, 32, config["d_model"])

    with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
        with record_function("model_forward"):
            output = model(src, tgt)

    filepath = f"data/raw/trace_{config['name']}_{trace_id}.json"
    prof.export_chrome_trace(filepath)
    return filepath

# test with one trace first
path = generate_trace(configs[0], 0)
print(f"✅ Test trace saved: {path}")

✅ Test trace saved: data/raw/trace_tiny_0.json


In [13]:
def parse_trace(filepath):
    with open(filepath) as f:
        trace = json.load(f)

    ops = [
        e for e in trace["traceEvents"]
        if e.get("cat") == "cpu_op" and "dur" in e
    ]
    ops = sorted(ops, key=lambda x: x["ts"])

    G = nx.DiGraph()
    for i, op in enumerate(ops):
        node_type = "network" if "allreduce" in op["name"].lower() or "comm" in op["name"].lower() else "compute"
        G.add_node(i,
            name=op["name"],
            type=node_type,
            duration_ms=op["dur"] / 1000,
            timestamp=op["ts"]
        )
        if i > 0:
            G.add_edge(i-1, i)
    return G

# test
G = parse_trace(path)
print(f"✅ Parser working: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

✅ Parser working: 574 nodes, 573 edges


In [14]:
def find_critical_path(G):
    for u, v in G.edges():
        G[u][v]['weight'] = G.nodes[u]['duration_ms']
    critical_path = nx.dag_longest_path(G, weight='weight')
    critical_path_length = nx.dag_longest_path_length(G, weight='weight')
    return critical_path, critical_path_length

def add_features(G, critical_path):
    total_time = sum(G.nodes[n]['duration_ms'] for n in G.nodes())
    max_dur = max(G.nodes[n]['duration_ms'] for n in G.nodes())
    for n in G.nodes():
        duration = G.nodes[n]['duration_ms']
        G.nodes[n]['compute_ratio'] = duration / total_time
        G.nodes[n]['is_critical'] = 1 if n in critical_path else 0
        G.nodes[n]['norm_duration'] = duration / max_dur
    return G

# test
critical_path, total_time = find_critical_path(G)
G = add_features(G, critical_path)
print(f"✅ Features added. Total time: {total_time:.2f}ms")

✅ Features added. Total time: 386.75ms


In [15]:
def to_pyg(G):
    compute_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'compute']
    network_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'network']

    data = HeteroData()
    data['compute'].x = torch.tensor([
        [G.nodes[n]['duration_ms'],
         G.nodes[n]['compute_ratio'],
         G.nodes[n]['norm_duration'],
         G.nodes[n]['is_critical']]
        for n in compute_nodes
    ], dtype=torch.float)

    data['network'].x = torch.tensor([
        [G.nodes[n]['duration_ms'],
         G.nodes[n]['compute_ratio'],
         G.nodes[n]['norm_duration'],
         G.nodes[n]['is_critical']]
        for n in network_nodes
    ], dtype=torch.float) if network_nodes else torch.zeros((0, 4))

    return data

# test
data = to_pyg(G)
print(f"✅ PyG working: compute={data['compute'].x.shape}")

✅ PyG working: compute=torch.Size([574, 4])


In [16]:
all_graphs = []

for config in configs:
    for trace_id in range(167):  # ~500 total across 3 configs
        filepath = generate_trace(config, trace_id)
        G = parse_trace(filepath)
        critical_path, total_time = find_critical_path(G)
        G = add_features(G, critical_path)
        data = to_pyg(G)

        # save pyg graph
        torch.save(data, f"data/graphs/graph_{config['name']}_{trace_id}.pt")
        all_graphs.append(data)

        if trace_id % 50 == 0:
            print(f"✅ {config['name']} — {trace_id}/167 done")

print(f"\n🎉 Total graphs generated: {len(all_graphs)}")

✅ tiny — 0/167 done
✅ tiny — 50/167 done
✅ tiny — 100/167 done
✅ tiny — 150/167 done
✅ small — 0/167 done
✅ small — 50/167 done
✅ small — 100/167 done
✅ small — 150/167 done
✅ medium — 0/167 done
✅ medium — 50/167 done
✅ medium — 100/167 done
✅ medium — 150/167 done

🎉 Total graphs generated: 501
